# 🐴 競馬予測モデル学習

**使い方：**
1. セル①でライブラリインストール
2. セル②でSupabase接続
3. セル③でデータ取得
4. セル④で特徴量エンジニアリング
5. セル⑤でLightGBM学習
6. セル⑥でモデル保存

In [ ]:
# ① ライブラリインストール
!pip install supabase lightgbm scikit-learn matplotlib seaborn -q

In [ ]:
# ② Supabase接続
SUPABASE_URL = 'https://infypumigexmpdmijhnx.supabase.co'
SUPABASE_KEY = 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9...'  # service_role keyを貼り付け

from supabase import create_client
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Supabase接続OK')

In [ ]:
# ③ 特徴量データ取得（v_featuresビューから）
import pandas as pd

# ページネーションで全件取得
all_data = []
offset = 0
while True:
    res = supabase.table('v_features').select('*').range(offset, offset+999).execute()
    if not res.data: break
    all_data.extend(res.data)
    offset += 1000
    print(f'取得済み: {len(all_data)}件', end='\r')

df = pd.DataFrame(all_data)
df['race_date'] = pd.to_datetime(df['race_date'])
print(f'\n✅ データ取得完了: {len(df):,}件')

In [ ]:
# ④ 特徴量エンジニアリング・前処理
from sklearn.preprocessing import LabelEncoder

# カテゴリ変数をエンコード
cat_cols = ['venue_code', 'surface', 'track_condition', 'weather', 'class']
le = {}
for col in cat_cols:
    le[col] = LabelEncoder()
    df[col + '_enc'] = le[col].fit_transform(df[col].astype(str))

# 特徴量リスト
feature_cols = [
    # レース条件
    'venue_code_enc', 'distance', 'surface_enc', 'track_condition_enc',
    'weather_enc', 'post_position', 'entry_count', 'class_enc',
    # 馬の過去成績
    'avg_finish_3', 'avg_finish_5', 'fukusho_rate_3', 'fukusho_rate_5',
    'last_finish', 'rest_days', 'distance_diff', 'horse_weight', 'horse_weight_diff',
    # 騎手・調教師
    'jockey_fukusho_rate', 'trainer_fukusho_rate',
    # 血統
    'father_fukusho_rate', 'mother_father_fukusho_rate',
    # オッズ・人気
    'odds', 'popularity', 'popularity_ratio',
]

X = df[feature_cols]
y = df['target']

print(f'✅ 特徴量数: {len(feature_cols)}')

In [ ]:
# ⑤ 時系列分割 & LightGBM学習
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# 時系列分割
train_mask = df['race_date'] < '2025-01-01'
valid_mask = (df['race_date'] >= '2025-01-01') & (df['race_date'] < '2026-01-01')
test_mask  = df['race_date'] >= '2026-01-01'

X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'学習: {len(X_train):,}件 | 検証: {len(X_valid):,}件 | テスト: {len(X_test):,}件')

# パラメータ
params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'min_child_samples': 20,
    'class_weight': 'balanced',
    'verbose': -1,
}

# 学習
model = lgb.LGBMClassifier(**params, n_estimators=1000)
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# 評価
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f'\n✅ AUC: {auc:.4f}')

In [ ]:
# ⑥ 特徴量重要度の確認
import matplotlib.pyplot as plt

lgb.plot_importance(model, max_num_features=20, figsize=(10, 8))
plt.title('特徴量重要度')
plt.tight_layout()
plt.show()

In [ ]:
# ⑦ モデル保存
import joblib
from datetime import datetime

date_str = datetime.now().strftime('%Y%m%d')
fname = f'keiba_model_{date_str}.pkl'
joblib.dump(model, fname)
print(f'✅ モデル保存: {fname}')

# Google Driveに保存する場合
# from google.colab import drive
# drive.mount('/content/drive')
# joblib.dump(model, f'/content/drive/MyDrive/keiba/{fname}')